# Smith v. United States — GPT-5.5 (3rd round, 2026-05)

Latest OpenAI flagship (`gpt-5.5`, released April 2026, became ChatGPT default 2026-05-05). Continues the methodology of `smith_chat_gpt.ipynb` / `smith_chat_gpt_2.ipynb` and uses the same prompts, parser, and 100-completion sampling so the new data lines up with the older runs in `analyze_responses.ipynb`.

**Temperature is intentionally not passed.** The OpenAI docs for `gpt-5.5` do not confirm `temperature` as a supported parameter (recent reasoning-capable flagships often reject it); to avoid breaking the 100-call run, we rely on the model's default sampling. If the user later confirms `temperature` is accepted, set it explicitly here.

API keys are loaded from `.env` at the repo root via `python-dotenv`.

## Setup
Load libraries and the `.env` file (which lives at the repo root, one level above `code/`).

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import pandas as pd

load_dotenv(os.path.join('..', '.env'))

True

## Model + client
The family alias is pinned in `MODEL`. The probe cell below captures the exact dated snapshot the API resolves it to, so the precise micro-version used for the run is preserved in the notebook output.

In [2]:
# Latest GPT-5.5 flagship (OpenAI) as of 2026-05-06.
# Dated snapshot at time of writing: gpt-5.5-2026-04-23.
# We pin the family alias; the exact dated snapshot the API resolves to is
# captured in the probe cell below (response.model).
MODEL = 'gpt-5.5'

# OpenAI() picks up OPENAI_API_KEY from the environment.
client = OpenAI()

In [3]:
def get_completion(prompt: str) -> str:
    '''Generate a GPT-5.5 chat completion for `prompt`.'''
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.choices[0].message.content

In [4]:
# Probe call — captures the exact dated snapshot the API resolves the alias to.
probe = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with one word: ok'}],
)
print(f'Requested model: {MODEL}')
print(f'Resolved model:  {probe.model}')
print(f'Probe content:   {probe.choices[0].message.content!r}')

Requested model: gpt-5.5
Resolved model:  gpt-5.5-2026-04-23
Probe content:   'ok'


## Inputs
Load the statute and the *Smith* case summary. The notebook is run from `code/`, so we reach `data/` via `os.path.join('..', 'data')`.

In [5]:
DATAPATH = os.path.join('..', 'data')

statute = open(os.path.join(DATAPATH, '18USC924c1.txt'), 'r').read()
case_summary = open(os.path.join(DATAPATH, 'smith_case_summary.txt'), 'r').read()

## Prompt with visible analysis (`prompt`)
First variant: model is asked to *show* its reasoning inside `<legal_interpretation>` tags before giving the structured answer. Used here once for sanity / qualitative inspection — **not** used in the 100-call loop.

In [6]:
prompt = f'''
You are asked to consider a legal scenario from the perspective of an ordinary person, not a legal expert. Your task is to read a statute and a case summary, then decide whether the actions described constitute 'use' of a gun as defined by the statute.

First, please read the following statute:

<statute>
{statute}
</statute>

Now, read the following case summary:

<case_summary>
{case_summary}
</case_summary>

Before providing your final answer, please analyze the situation in detail inside <legal_interpretation> tags. Consider the facts of the case and how they relate to the language of the statute. Remember, you're approaching this as an average person, not a legal professional.

In your analysis, include the following:
1. Quote the relevant parts of the statute that define 'use' of a gun.
2. List the key actions Mr. Smith took with the gun, according to the case summary. Number each action.
3. Explain how an ordinary person might interpret these actions in relation to the statute.
4. Identify any aspects of the case that might influence your interpretation.
5. Present arguments for and against considering Mr. Smith's actions as 'use' of the gun.
6. Form your opinion on whether a reasonable person would consider that Mr. Smith 'used' his gun in the sense that the language of the statute requires.

Based on your analysis, please provide your answer in the following format:

ANSWER: [Your answer (yes or no)]
REASONING: [Your reasoning, explained in simple terms that an average person would use]

Please provide your response now.
'''

In [7]:
response = get_completion(prompt)

In [8]:
display(Markdown(response))

<legal_interpretation>

1. **Relevant parts of the statute**

The statute does not give a separate definition of “use.” The relevant language is:

- “Whoever, during and in relation to any crime of violence or drug trafficking crime … **uses or carries a firearm** …”
- “the crime of violence or drug trafficking crime in which the firearm was **used or carried**.”

So the key question is whether Mr. Smith’s actions count as “uses” a firearm “during and in relation to” a drug trafficking crime.

2. **Key actions Mr. Smith took with the gun**

1. Mr. Smith had an automatic MAC–10 firearm and silencer with him during a cocaine-buying trip.
2. He discussed the firearm with a drug dealer.
3. He said he might sell the gun if another possible deal fell through.
4. He later told an undercover officer that he had an automatic MAC–10 and silencer.
5. He pulled the MAC–10 out of a bag and showed it to the officer.
6. When asked what he wanted for the gun, he asked for drugs instead of money.
7. He offered to trade the MAC–10 for two ounces of cocaine.

3. **How an ordinary person might interpret these actions**

An ordinary person might think “use a gun” usually means firing it, threatening someone with it, pointing it, or using it for protection. Mr. Smith did not fire the gun or threaten anyone with it.

But an ordinary person might also say that if someone uses a gun as something to trade for cocaine, he is still “using” the gun—just not as a weapon. He is using it as payment or as an item of value in a drug deal.

4. **Aspects that might influence the interpretation**

- The gun was not just present; Mr. Smith actively brought it out and offered it in exchange for cocaine.
- The proposed exchange was directly connected to drugs.
- The gun was a dangerous automatic MAC–10 with a silencer, which makes its presence more serious.
- The statute says “uses or carries,” not “fires,” “brandishes,” or “threatens with,” so “use” could be understood broadly.
- On the other hand, because the statute is about firearms during violent or drug crimes, some people might think “use” means using the gun as a weapon, not as merchandise.

5. **Arguments for and against calling this “use”**

**For “use”:**
- Mr. Smith used the gun as an item of barter to try to obtain cocaine.
- The gun played an active role in the drug transaction.
- He did more than merely possess it; he displayed it and offered it as payment.
- In everyday language, people say someone “used” something if they used it to get something else.

**Against “use”:**
- He did not fire the gun.
- He did not point it at anyone or threaten anyone.
- He was using it more like property or merchandise, not as a firearm in the usual weapon sense.
- Some ordinary people might think the statute is aimed at guns being used for intimidation or violence during crimes.

6. **Opinion**

A reasonable ordinary person could conclude that Mr. Smith “used” the gun because he made it part of the drug deal. Even though he did not use it as a weapon, he used it as a thing of value to obtain cocaine. That seems to fit the broad ordinary meaning of “use.”

</legal_interpretation>

ANSWER: Yes

REASONING: Mr. Smith did not shoot or threaten anyone with the gun, but he did more than simply have it nearby. He pulled out the MAC–10, showed it to the undercover officer, and offered to trade it for cocaine. In ordinary language, that means he “used” the gun as payment in a drug deal. So a reasonable person could say he used the firearm during and in relation to a drug trafficking crime.

## Constrained prompt (`new_prompt`)
Second variant: same instructions, but the analytical steps are silent. The response is constrained to `ANSWER: <yes/no>` followed by `REASONING: <text>` so the parser can split cleanly. **This is the prompt used for the 100-call loop.**

In [9]:
new_prompt = f'''
You are asked to consider a legal scenario from the perspective of an ordinary person, not a legal expert. Your task is to read a statute and a case summary, then decide whether the actions described constitute 'use' of a gun as defined by the statute.

First, please read the following statute:

<statute>
{statute}
</statute>

Now, read the following case summary:

<case_summary>
{case_summary}
</case_summary>

Before providing your final answer, please analyze the situation in detail. Consider the facts of the case and how they relate to the language of the statute. Remember, you're approaching this as an average person, not a legal professional.

Specifically, consider the following (but DO NOT include them in your response):
1. The relevant parts of the statute that define 'use' of a gun.
2. The key actions Mr. Smith took with the gun, according to the case summary. Number each action.
3. How an ordinary person might interpret these actions in relation to the statute.
4. Any aspects of the case that might influence your interpretation.
5. Arguments for and against considering Mr. Smith's actions as 'use' of the gun.
6. Your opinion on whether a reasonable person would consider that Mr. Smith 'used' his gun in the sense that the language of the statute requires.

Based on your analysis, please provide your answer in the following format:

ANSWER: [Your answer (yes or no)]
REASONING: [Your reasoning, explained in simple terms that an average person would use]

Please provide your response now.
'''

In [10]:
test_response = get_completion(new_prompt)
display(Markdown(test_response))

ANSWER: Yes

REASONING: Even though Mr. Smith did not fire the gun, point it at anyone, or threaten anyone with it, he did more than merely possess it. He took out the MAC–10, showed it to the undercover officer, and offered to trade it for cocaine. In ordinary terms, he was using the gun as something valuable to help get drugs—basically using it as payment or bargaining power in a drug deal.

A person could argue that “use” of a gun means using it as a weapon, but the statute is about using or carrying a firearm during and in relation to a drug trafficking crime. Here, the gun was directly tied to the drug transaction because Smith offered it in exchange for cocaine. So, an average person could reasonably say he “used” the gun within the meaning of the statute.

## Quick 3-iteration parser check
Make sure the `ANSWER:` / `REASONING:` parsing handles the model's output before committing to 100 paid calls.

In [11]:
test_responses = []
test_answers = []
test_reasoning = []

for i in range(3):
    t_response = get_completion(new_prompt)
    test_answers.append(t_response[t_response.find('ANSWER:') + 7:t_response.find('REASONING:')].strip())
    test_reasoning.append(t_response[t_response.find('REASONING:') + 10:].strip())
    test_responses.append(t_response)

test_df = pd.DataFrame({'response': test_responses, 'answer': test_answers, 'reasoning': test_reasoning})
display(test_df)

,response,answer,reasoning
0,ANSWER: Yes\n\nREASONING: Mr. Smith did more t...,Yes,Mr. Smith did more than simply own or possess ...
1,ANSWER: Yes\n\nREASONING: Mr. Smith did more t...,Yes,Mr. Smith did more than simply have the gun ne...
2,ANSWER: yes\n\nREASONING: Mr. Smith did more t...,yes,Mr. Smith did more than merely possess the gun...


## Generate 100 completions and save
The expensive cell. Hits the API 100 times with no rate-limit handling — expect minutes of wall time.

In [12]:
answers = []
reasoning = []

for i in range(100):
    response = get_completion(new_prompt)
    answer_start = response.find('ANSWER:') + 7
    reasoning_start = response.find('REASONING:') + 10
    answer = response[answer_start:response.find('REASONING:')].strip()
    reason = response[reasoning_start:].strip()
    answers.append(answer)
    reasoning.append(reason)

responses_df = pd.DataFrame({'answer': answers, 'reasoning': reasoning})
display(responses_df.head())

,answer,reasoning
0,Yes,Mr. Smith did more than just have the gun near...
1,Yes,Mr. Smith did not fire the gun or threaten any...
2,Yes,Mr. Smith did not just happen to have the gun ...
3,Yes,An ordinary person could reasonably say Mr. Sm...
4,Yes,Mr. Smith did more than just own or possess th...


## Inspect the answer distribution
If any answers come back wrapped in markdown (e.g. `Yes**\n\n**`), normalize them with a `.loc[...]` fix-up here — see the `smith_chat_gpt_2.ipynb` cell 13 pattern.

In [14]:
responses_df['answer'] = responses_df['answer'].str.strip().str.capitalize()
print(responses_df['answer'].value_counts())

answer
Yes    100
Name: count, dtype: int64


## Save
Write the cleaned DataFrame to `data/`. The CSV name encodes the exact model family for downstream `analyze_responses.ipynb` consumption.

In [15]:
OUT_CSV = os.path.join(DATAPATH, 'smith_responses_gpt_5_5.csv')
responses_df.to_csv(OUT_CSV, index=False)
print(f'Wrote {len(responses_df)} rows to {OUT_CSV}')

Wrote 100 rows to ../data/smith_responses_gpt_5_5.csv
